# Visualise peak and event tables

AudioSpylt tables are TSV files with a kind header. This notebook reads the
example files under `data/tables/`, plots snapshots and events, then upgrades a
two-row legacy table to v1. No prior tutorial is required.

In [ ]:
from pathlib import Path
import pandas as pd
import audiospylt as asp

## Table files

The first line of every written table is `# audiospylt-table: <kind> v1`. The
three kinds are `snapshot` (one frame of `freq` / `amp`), `events` (time
segments with start/stop frequency and amplitude), and `tracks` (partial
trajectories with a `track_id`). `infer_kind` guesses from the columns;
`validate` checks a declared kind.

Before you run the cell: what kind does `infer_kind` report for `speech1.tsv`
and for `sigmoid_high.tsv`?

In [ ]:
header = Path("../data/tables/speech1.tsv").read_text().splitlines()[0]
print(header)
speech = asp.read_table("../data/tables/speech1.tsv")
conga = asp.read_table("../data/tables/conga3.tsv")
events_hi = asp.read_table("../data/tables/sigmoid_high.tsv")
print("speech:", asp.infer_kind(speech), f"({len(speech)} rows)")
print("sigmoid_high:", asp.infer_kind(events_hi), f"({len(events_hi)} rows)")
asp.validate(speech, "snapshot")
asp.validate(events_hi, "events")
print(f"speech {speech.freq.min():.1f}–{speech.freq.max():.1f} Hz")
print(f"conga {conga.freq.min():.1f}–{conga.freq.max():.1f} Hz")
speech.head()

`speech1` is a snapshot (41 peaks, 97.4–3055.3 Hz); `sigmoid_high` is events
(41 rows). `conga3` spans 31.2–600.0 Hz. Both kinds validate.

## Snapshots

`plot_snapshot` overlays two peak lists. `freq_scale="log"` spreads the low
partials. Compare the printed ranges: speech reaches above 3 kHz; conga stays
below 600 Hz.

In [ ]:
fig = asp.viz.plot_snapshot(speech, conga, labels=["speech1", "conga3"], freq_scale="log")
asp.nb.show(fig)
print(f"speech {len(speech)} peaks, conga {len(conga)} peaks")

## Events in 2-D

Each row of an events table is a line from `(time_start, freq_start)` to
`(time_stop, freq_stop)`. `sigmoid_high` and `sigmoid_low` are two morph
results from the `sigmoid` method (high vs low parameter), each with 41 rows.
The plot compares how their frequencies move over the same schedule.

In [ ]:
events_lo = asp.read_table("../data/tables/sigmoid_low.tsv")
fig = asp.viz.plot_events(events_hi, events_lo, labels=["sigmoid_high", "sigmoid_low"])
asp.nb.show(fig)
print(f"rows: high={len(events_hi)}, low={len(events_lo)}")

## Events in 3-D

`plot_events_3d` adds amplitude, in dB, as a third axis: time runs left to
right and frequency into depth. The faint lines on the floor are the paths of
the 2-D plot above. Drag to turn the box; hover a line to read time,
frequency, amplitude and pitch. `db=False`, `shadow=False` and
`freq_scale="log"` change the view. Before you run: do the two tables still
both show 41 segments in 3-D?

In [ ]:
fig = asp.viz.plot_events_3d(events_hi, events_lo, labels=["sigmoid_high", "sigmoid_low"])
asp.nb.show(fig)
print(f"3-D rows: high={len(events_hi)}, low={len(events_lo)}")

## Upgrade a legacy table

A pre-v1 TSV has a column header row (`Frequency (Hz)`, `Amplitude`) but no
`# audiospylt-table` line. `read_table` renames the columns, validates, and
emits one `LegacyFormatWarning`. `upgrade` does the same rename and validate on
an in-memory frame (a second warning). `write_table` then stores the v1 header.

In [ ]:
Path("tsv").mkdir(exist_ok=True)
legacy = pd.DataFrame({"Frequency (Hz)": [440.0, 880.0], "Amplitude": [0.5, 0.25]})
legacy_path = Path("tsv/legacy_peaks.tsv")
legacy.to_csv(legacy_path, sep="\t", index=False)
upgraded = asp.read_table(legacy_path)
same = asp.upgrade(legacy)
v1_path = asp.write_table(upgraded, "tsv/legacy_peaks_v1.tsv")
print(v1_path.read_text().splitlines()[0])
print(f"upgrade equals read_table: {same.equals(upgraded)}")
upgraded

Each call warns once about the old columns. The written file starts with
`# audiospylt-table: snapshot v1`, and both results hold `freq` / `amp` with the
same two values (440 Hz at 0.5, 880 Hz at 0.25).

## A schema failure

`validate` reports every broken column. A negative amplitude is rejected.

In [ ]:
bad = pd.DataFrame({"freq": [440.0], "amp": [-0.5]})
try:
    asp.validate(bad, "snapshot")
except asp.SchemaError as e:
    print(e)

The message is `Invalid table:` then `- amp: must be >= 0 (1 row(s); first row
positions [0])`.